# Neural Network Time Series Forecasting

## Objective

The goal of this notebook is to forecast daily unit sales using a neural network approach based on historical sales patterns and engineered time-series features.

The model uses:

- lag features,
- rolling statistics,
- calendar variables,
- and external signals such as oil prices.

The forecasting performance is evaluated on the January–March 2014 test period using standard forecasting metrics.


## Step 1 — Imports and Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import torch
import torch.nn as nn

print("All libraries imported successfully!")

## Step 2 — Load Prepared Dataset

This notebook uses the prepared `timeseries_with_features.csv` dataset generated during the feature engineering stage.


In [ ]:
# Detect project root

current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR = PROJECT_ROOT / "data"

DATA_PATH = OUTPUTS_DIR / "timeseries_with_features.csv"

if not DATA_PATH.exists():
    DATA_PATH = DATA_DIR / "timeseries_with_features.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find timeseries_with_features.csv in outputs/ or data/"
    )

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"],
    index_col="date"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "to", df.index.max().date())

df.head()

## Step 3 — Chronological Train/Test Split

The dataset is split chronologically to avoid data leakage.

- Training period: before January 1, 2014
- Test period: January 1, 2014 to March 31, 2014


In [ ]:
train_cutoff = "2014-01-01"

df_train = df[df.index < train_cutoff].copy()
df_test = df[df.index >= train_cutoff].copy()

print("Training rows:", len(df_train))
print("  From:", df_train.index.min().date(), "to", df_train.index.max().date())

print("\nTest rows:", len(df_test))
print("  From:", df_test.index.min().date(), "to", df_test.index.max().date())

## Step 4 — Feature Selection and Validation

Neural networks require numeric input variables.  
The weekday column is converted from text labels into numeric values.


In [ ]:
# Convert day_of_week from text to numeric

day_mapping = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6,
}

df["day_of_week_num"] = df["day_of_week"].map(day_mapping)

# Re-create train/test after adding the new column
df_train = df[df.index < train_cutoff].copy()
df_test = df[df.index >= train_cutoff].copy()

features = [
    "year",
    "month",
    "day_number",
    "day_of_week_num",
    "is_weekend",

    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",

    "rolling_7d_mean",
    "rolling_14d_mean",
    "rolling_30d_mean",
    "rolling_7d_std",

    "dcoilwtico",
    "oil_lag_1",
    "oil_rolling_7d_mean",
]

target = "unit_sales"

required_columns = features + [target]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]
y_test = df_test[target]

print("Feature validation passed.")
print("Number of features:", len(features))
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## Step 5 — Feature Scaling

Neural networks are sensitive to feature magnitudes, so the input variables are standardized before training.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")

## Step 6 — Sequence Preparation

The neural network learns from rolling historical windows.

Each sequence contains the previous 7 days of feature values and predicts the next day of unit sales.


In [ ]:
SEQUENCE_LENGTH = 7

def create_sequences(X, y, seq_length):
    X_seq = []
    y_seq = []

    for i in range(seq_length, len(X)):
        X_seq.append(X[i-seq_length:i])
        y_seq.append(y[i])

    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train.values,
    SEQUENCE_LENGTH
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test.values,
    SEQUENCE_LENGTH
)

print("Training sequence shape:", X_train_seq.shape)
print("Test sequence shape:", X_test_seq.shape)

## Step 7 — Build Neural Network

This notebook uses a simple LSTM regression model.

The LSTM reads a sequence of recent observations and outputs a single sales forecast.


In [ ]:
class LSTMRegressor(nn.Module):

    def __init__(self, input_size, hidden_size=64):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out


model = LSTMRegressor(
    input_size=X_train_seq.shape[2],
    hidden_size=64
)

print(model)

## Step 8 — Train Model

The model is trained on the chronological training dataset.

A moderate number of epochs is used to keep the notebook lightweight and reproducible.


In [ ]:
torch.manual_seed(42)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

X_train_tensor = torch.tensor(
    X_train_seq,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_seq,
    dtype=torch.float32
).view(-1, 1)

EPOCHS = 50

loss_history = []

for epoch in range(EPOCHS):

    model.train()

    predictions = model(X_train_tensor)

    loss = criterion(
        predictions,
        y_train_tensor
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss.item():.4f}"
        )

print("Model training complete!")

## Step 9 — Training Loss Visualization

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(loss_history)

plt.title("Neural Network Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 10 — Generate Predictions

In [ ]:
model.eval()

X_test_tensor = torch.tensor(
    X_test_seq,
    dtype=torch.float32
)

with torch.no_grad():
    nn_predictions = (
        model(X_test_tensor)
        .numpy()
        .flatten()
    )

# Align df_test with the sequence output length
df_test_eval = df_test.iloc[SEQUENCE_LENGTH:].copy()

df_test_eval["nn_prediction"] = nn_predictions

print("Predictions added.")
print("Evaluation rows:", len(df_test_eval))

df_test_eval[["unit_sales", "nn_prediction"]].head()

## Step 11 — Model Evaluation

The neural network predictions are evaluated using several forecasting metrics:

- MAE
- RMSE
- MAPE
- Bias
- R²


In [ ]:
y_true = df_test_eval[target].values
y_pred = df_test_eval["nn_prediction"].values

non_zero_mask = y_true != 0

mae = mean_absolute_error(y_true, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_true, y_pred)
)

mape = (
    np.mean(
        np.abs(
            (y_true[non_zero_mask] - y_pred[non_zero_mask])
            / y_true[non_zero_mask]
        )
    ) * 100
)

bias = np.mean(y_true - y_pred)

r2 = r2_score(y_true, y_pred)

metrics = pd.DataFrame({
    "model": ["Neural Network"],
    "MAE": [mae],
    "RMSE": [rmse],
    "MAPE": [mape],
    "Bias": [bias],
    "R2": [r2]
})

display(metrics.round(3))

## Model Interpretation

The evaluation metrics help compare the forecasting performance of the neural network against the statistical models and other machine learning approaches developed throughout the project.

In general, stronger forecasting models tend to have:

- lower MAE,
- lower RMSE,
- lower MAPE,
- and Bias values close to zero.


## Step 12 — Actual vs Predicted Sales

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    df_test_eval.index,
    df_test_eval["unit_sales"],
    label="Actual Sales"
)

plt.plot(
    df_test_eval.index,
    df_test_eval["nn_prediction"],
    label="NN Prediction"
)

plt.title("Actual vs Predicted Unit Sales")
plt.xlabel("Date")
plt.ylabel("Unit Sales")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 13 — Forecast Error Analysis

In [ ]:
df_test_eval["error"] = (
    df_test_eval["unit_sales"]
    - df_test_eval["nn_prediction"]
)

df_test_eval["abs_error"] = df_test_eval["error"].abs()

print("Forecast error summary:")
print(
    df_test_eval[["error", "abs_error"]]
    .describe()
    .round(2)
)

## Forecast Error Analysis

The forecast error analysis provides additional insight into the prediction quality of the neural network model.

Positive errors indicate under-prediction, while negative errors indicate over-prediction.

Some larger errors may remain during periods with abrupt demand changes or extreme sales spikes, which are common challenges in time-series forecasting.


## Step 14 — Residual Analysis

Residual analysis helps determine whether forecasting errors behave randomly over time.

Ideally, residuals should fluctuate around zero without displaying strong systematic patterns.


In [ ]:
df_test_eval["residual"] = (
    df_test_eval["unit_sales"]
    - df_test_eval["nn_prediction"]
)

plt.figure(figsize=(15, 5))

plt.plot(
    df_test_eval.index,
    df_test_eval["residual"]
)

plt.axhline(
    0,
    linestyle="--"
)

plt.title("Neural Network Residuals Over Time")
plt.xlabel("Date")
plt.ylabel("Residual")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))

plt.hist(
    df_test_eval["residual"],
    bins=30
)

plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

## Step 15 — Save Model and Outputs

The trained neural network model and its associated artifacts are saved for reuse in the MLflow tracking workflow and the Streamlit forecasting application.


In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Save model parameters
torch.save(
    model.state_dict(),
    MODELS_DIR / "nn_model.pth"
)

# Save feature columns
with open(
    MODELS_DIR / "nn_feature_columns.json",
    "w"
) as f:
    json.dump(features, f, indent=4)

# Save scaler
import pickle

with open(
    MODELS_DIR / "nn_scaler.pkl",
    "wb"
) as f:
    pickle.dump(scaler, f)

# Save metrics
metrics.to_csv(
    MODELS_DIR / "nn_metrics.csv",
    index=False
)

# Save forecast outputs
forecast_output = df_test_eval[
    [
        "unit_sales",
        "nn_prediction",
        "error",
        "abs_error",
        "residual"
    ]
].copy()

forecast_output.to_csv(
    OUTPUTS_DIR / "nn_forecast_output.csv"
)

print("Artifacts saved successfully.")
print("-", MODELS_DIR / "nn_model.pth")
print("-", MODELS_DIR / "nn_feature_columns.json")
print("-", MODELS_DIR / "nn_scaler.pkl")
print("-", MODELS_DIR / "nn_metrics.csv")
print("-", OUTPUTS_DIR / "nn_forecast_output.csv")

## Step 16 — Final Conclusion

The neural network model demonstrates the ability to capture nonlinear relationships in the sales time series using engineered temporal features.

The model is capable of learning short-term sales dynamics and general forecasting patterns, although some larger forecasting errors may remain during periods with abrupt demand changes.

Overall, this notebook extends the forecasting workflow with a deep learning approach and produces reusable artifacts for later MLflow tracking and Streamlit deployment.
